# Scree Plots & Standardisation - Hands-on Tutorial

In this notebook we'll work with PCA output to answer two practical questions:

1. **How many PCs should I keep?** - scree plots, cumulative variance, and why there's no single right answer
2. **Why must I standardise first?** - how features on wildly different scales can hijack PCA, and how the z-score fixes it

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

---
## Part 1: Scree Plots and Choosing k

We'll generate a 10-dimensional dataset where only a few dimensions carry
real structure. Your job: figure out how many PCs matter.

In [ ]:
# Generate 10D data: 3 strong dimensions + 7 noisy ones
np.random.seed(5)
n = 200

# Three meaningful axes with different strengths, plus low-level noise
meaningful = np.random.randn(n, 3) @ np.diag([5, 2, 1])
noise = np.random.randn(n, 7) * 0.3
data = np.hstack([meaningful, noise])

# Centre the data
data_mean = data.mean(axis=0)
centred = data - data_mean

print(f"Data shape: {data.shape}  (n_samples, n_features)")
print(f"We know there are 3 meaningful dims — but pretend you don't.")

### Step 1: Eigendecomposition

Compute the covariance matrix and its eigenvalues/eigenvectors.
Sort in descending order (largest eigenvalue first).

In [ ]:
# YOUR CODE HERE
# 1. Compute the covariance matrix: np.cov(centred, rowvar=False)
# 2. Eigendecompose: np.linalg.eigh(cov_matrix)
# 3. Sort descending (eigh returns ascending order)
#    Hint: order = np.argsort(eigvals)[::-1]
# Store results in: cov_matrix, eigvals, eigvecs

raise NotImplementedError("Eigendecomposition")

### Step 2: Plot the scree plot

A scree plot shows the eigenvalue of each PC. Make two subplots side by side:
- **Left**: eigenvalue vs component number (the scree plot)
- **Right**: cumulative explained variance (%) vs number of components

The explained variance of PC $i$ is $\lambda_i / \sum_j \lambda_j$.

In [ ]:
# YOUR CODE HERE
# 1. Compute explained variance per component:
#    explained = eigvals / eigvals.sum() * 100
# 2. Compute cumulative explained variance:
#    cumulative = np.cumsum(explained)
# 3. Make a (1, 2) subplot figure:
#    - Left: plot component number (1..10) vs eigenvalue, use 'o-' markers
#    - Right: plot number of components (1..10) vs cumulative variance (%)
#      Add a horizontal dashed line at 95% for reference

raise NotImplementedError("Scree plot")

### Think about it

- Where does the scree plot "elbow" - where does it flatten out?
- How many components do you need to reach 95% cumulative variance?
- Based on the plot, how many PCs would you keep?

Notice this only worked because every feature was already on the same scale -
next we'll see what happens when they aren't.

---
## Part 2: Why You Must Standardise

PCA ranks directions by variance - but variance depends on a feature's *units*.
A feature measured in large numbers will have large variance and dominate the PCs,
even if it carries no real structure. Let's watch that happen, then fix it.

In [ ]:
# Three measured features on very different scales:
rng = np.random.default_rng(0)
n = 300

# A hidden "activity" factor we'd like PCA to recover
g = rng.normal(0, 1, n)

# Two SMALL-range features that both track g -- this is the real structure
flux_mJy = 0.005 + 0.0015 * g + rng.normal(0, 0.0005, n)   # ~0.001 - 0.01
color    = 1.0   + 0.4    * g + rng.normal(0, 0.15,  n)    # ~0 - 2

# A LARGE-range feature that is pure scatter, unrelated to g
size_arcsec = rng.uniform(1, 100, n)                       # big numbers, no structure

X = np.column_stack([flux_mJy, color, size_arcsec])
feat_names = ['flux_mJy', 'color', 'size_arcsec']

print(f"{'feature':12s} {'min':>10s} {'max':>10s} {'variance':>12s}")
for name, col in zip(feat_names, X.T):
    print(f"{name:12s} {col.min():10.3g} {col.max():10.3g} {col.var():12.3g}")

### A small PCA helper, run on the raw data

We'll reuse this `pca` helper for both the raw and the standardised data, so the
only thing that changes between the two runs is the scaling. Run it on raw `X` first.

In [ ]:
def pca(M):
    """Centre M, return (eigvals, eigvecs, var_frac) sorted by descending eigenvalue."""
    Mc = M - M.mean(axis=0)
    cov = np.cov(Mc, rowvar=False)
    vals, vecs = np.linalg.eigh(cov)
    order = np.argsort(vals)[::-1]
    vals, vecs = vals[order], vecs[:, order]
    return vals, vecs, vals / vals.sum()

vals_raw, vecs_raw, vf_raw = pca(X)
print("Raw-data PCA, explained variance (%):", np.round(vf_raw * 100, 1))

plt.figure(figsize=(6, 3))
plt.bar(feat_names, vecs_raw[:, 0], color='#e74c3c')
plt.axhline(0, color='k', lw=0.6)
plt.ylabel('PC1 loading')
plt.title(f'Raw data: PC1 carries {vf_raw[0]*100:.0f}% of the variance')
plt.tight_layout(); plt.show()

### Step: standardise, then re-run PCA

Right now PC1 is almost entirely `size_arcsec` - not because it's informative, but
because it has the biggest numbers. Fix that by **standardising** each feature to
mean 0 and standard deviation 1 (the z-score), then run PCA again.

In [ ]:
# YOUR CODE HERE
# 1. Standardise each feature (column) to mean 0, std 1 -- the z-score:
#       mu    = X.mean(axis=0)
#       sigma = X.std(axis=0)
#       X_std = (X - mu) / sigma
# 2. Run PCA on the standardised data using the helper above:
#       vals_std, vecs_std, vf_std = pca(X_std)

raise NotImplementedError("Standardise the features, then run PCA on them")

In [ ]:
# Compare PC1's loadings before and after standardising
fig, axes = plt.subplots(1, 2, figsize=(11, 3.5), gridspec_kw={'wspace': 0.35})
axes[0].bar(feat_names, vecs_raw[:, 0], color='#e74c3c')
axes[0].set_title(f'Raw: PC1 ({vf_raw[0]*100:.0f}% var)')
axes[0].set_ylabel('PC1 loading')
axes[1].bar(feat_names, vecs_std[:, 0], color='#27ae60')
axes[1].set_title(f'Standardised: PC1 ({vf_std[0]*100:.0f}% var)')
for ax in axes:
    ax.axhline(0, color='k', lw=0.6)
    ax.tick_params(axis='x', rotation=20)
plt.tight_layout(); plt.show()

### Think about it

- On the raw data, PC1's loading points almost entirely at `size_arcsec`. Is that
  feature the most *informative*, or just the largest-numbered?
- After standardising, PC1 blends `flux_mJy` and `color` - the two features that
  share the hidden factor `g`. PCA could only see that structure once the features
  were on a common scale.
- Standardising made `size_arcsec` *less* dominant. When might that be the wrong
  move - i.e. when is a feature's raw scale genuinely meaningful and worth keeping?